In [1]:
import os
from dotenv import load_dotenv
from google.adk.agents import LlmAgent, TeamAgent

load_dotenv()

# 1. The Expert Researcher
researcher = LlmAgent(
    name="Researcher",
    instruction="You provide deep technical insights on topics.",
    model="gemini-3-flash"
)

# 2. The Editor
editor = LlmAgent(
    name="Editor",
    instruction="You take technical research and make it easy for a CEO to read.",
    model="gemini-3-flash"
)

# 3. The Team (Multi-Agent System)
my_team = TeamAgent(
    name="TechInsights",
    members=[researcher, editor],
    manager_instruction="Coordinate the researcher and editor to provide a high-level briefing."
)

# Run a test
if __name__ == "__main__":
    response = my_team.run("Explain why we should use Gemini 3 for multi-agent systems.")
    print(response.text)

ImportError: cannot import name 'TeamAgent' from 'google.adk.agents' (c:\Users\Paras Gulati\VS Code Python\ADK_research\.venv\Lib\site-packages\google\adk\agents\__init__.py)

In [6]:
# ... (rest of your imports and agent definitions remain the same)

# 3. The Multi-Agent Workflow
my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[researcher, editor]  # Changed 'agents' to 'sub_agents'
)

if __name__ == "__main__":
    print("🚀 Agents are working...")
    # Using 'uv run' will handle the execution
    response = my_team.run("Explain why Gemini 3 is great for agents.")
    print(response.text)

ValidationError: 1 validation error for SequentialAgent
  Value error, Agent `Researcher` already has a parent agent, current parent: `TechInsightsTeam`, trying to add: `TechInsightsTeam` [type=value_error, input_value={'name': 'TechInsightsTea...l_error_callback=None)]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [7]:
import os
from dotenv import load_dotenv
from google.adk.agents import LlmAgent, SequentialAgent

load_dotenv()

# 1. The Expert Researcher
# Note: 'description' is used by the system to understand the agent's role in a team
researcher = LlmAgent(
    name="Researcher",
    description="Finds technical facts and data about AI models.",
    instruction="You provide deep technical insights. Focus on 3 key technical points.",
    model="gemini-3-flash"
)

# 2. The Editor
editor = LlmAgent(
    name="Editor",
    description="Polishes technical text for executive readers.",
    instruction="Summarize the technical research into a short, executive-style briefing.",
    model="gemini-3-flash"
)

# 3. The Multi-Agent Workflow
my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[researcher, editor] 
)

if __name__ == "__main__":
    print("🚀 Agents are working...")
    # Passing the prompt to the team
    response = my_team.run("Explain why Gemini 3 is great for agents.")
    
    # Check if response has text attribute (standard for ADK)
    if hasattr(response, 'text'):
        print("\n--- FINAL OUTPUT ---\n")
        print(response.text)
    else:
        print(response)

🚀 Agents are working...


AttributeError: 'SequentialAgent' object has no attribute 'run'

In [8]:
import os
from dotenv import load_dotenv
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner  # <--- New Import

load_dotenv()

# 1. The Expert Researcher
researcher = LlmAgent(
    name="Researcher",
    description="Finds technical facts about AI models.",
    instruction="List 3 key technical features of Gemini 3.",
    model="gemini-3-flash"
)

# 2. The Editor
editor = LlmAgent(
    name="Editor",
    description="Summarizes technical facts.",
    instruction="Turn the research into a one-sentence summary for a developer.",
    model="gemini-3-flash"
)

# 3. The Sequential Team
my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[researcher, editor] 
)

if __name__ == "__main__":
    print("🚀 Agents are starting via Runner...")
    
    # We initialize a Runner to execute our workflow
    runner = Runner()
    
    # Use runner.run() instead of my_team.run()
    result = runner.run(my_team, input="Start the research.")
    
    print("\n--- FINAL OUTPUT ---")
    print(result.text)

🚀 Agents are starting via Runner...


TypeError: Runner.__init__() missing 1 required keyword-only argument: 'session_service'

In [9]:
import os
import asyncio # New: ADK methods are often asynchronous
from dotenv import load_dotenv
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService # New: Session management

load_dotenv()

# 1. Define Agents (same as before)
researcher = LlmAgent(
    name="Researcher",
    description="Finds technical facts about AI models.",
    instruction="List 3 key technical features of Gemini 3.",
    model="gemini-3-flash"
)

editor = LlmAgent(
    name="Editor",
    description="Summarizes technical facts.",
    instruction="Turn the research into a one-sentence summary.",
    model="gemini-3-flash"
)

# 2. Define the Team
my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[researcher, editor] 
)

# 3. Execution Logic (Async)
async def main():
    # Initialize the Session Service
    session_service = InMemorySessionService()
    
    # Create the Runner with the required session_service
    runner = Runner(session_service=session_service)
    
    # ADK requires a session to exist before running
    # We give it a dummy user_id and session_id for local testing
    session = await session_service.create_session(
        app_name="ResearchApp", 
        user_id="local_user"
    )

    print("🚀 Agents are starting...")
    
    # Run the team within the created session
    result = await runner.run_async(
        my_team, 
        input="Explain Gemini 3 capabilities.",
        session_id=session.id
    )
    
    print("\n--- FINAL OUTPUT ---")
    print(result.text)

if __name__ == "__main__":
    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [10]:
import os
from dotenv import load_dotenv
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

# 1. Load your .env
load_dotenv()

# 2. Define Agents
researcher = LlmAgent(
    name="Researcher",
    description="Finds technical facts about AI models.",
    instruction="List 3 key technical features of Gemini 3.",
    model="gemini-3-flash"
)

editor = LlmAgent(
    name="Editor",
    description="Summarizes technical facts.",
    instruction="Turn the research into a one-sentence summary.",
    model="gemini-3-flash"
)

# 3. Define the Team
my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[researcher, editor] 
)

# 4. The Runner Logic
async def run_research():
    session_service = InMemorySessionService()
    runner = Runner(session_service=session_service)
    
    session = await session_service.create_session(
        app_name="ResearchApp", 
        user_id="local_user"
    )

    print("🚀 Agents are starting...")
    
    result = await runner.run_async(
        my_team, 
        input="Explain Gemini 3 capabilities.",
        session_id=session.id
    )
    
    return result.text

# 5. EXECUTE (In Jupyter, we await directly)
final_result = await run_research()
print("\n--- FINAL OUTPUT ---\n")
print(final_result)

ValueError: Either app or both app_name and agent must be provided.

In [11]:
async def run_research():
    # 1. Initialize services
    session_service = InMemorySessionService()
    
    # 2. FIXED: Pass the agent and app_name directly to the Runner
    runner = Runner(
        agent=my_team,             # Your SequentialAgent
        app_name="ResearchApp",    # Must match the session app_name
        session_service=session_service
    )
    
    # 3. Create a session
    session = await session_service.create_session(
        app_name="ResearchApp", 
        user_id="local_user"
    )

    print("🚀 Agents are starting...")
    
    # 4. Run the team
    # Note: Since the agent is already in the runner, we just provide the input
    result = await runner.run_async(
        input="Explain Gemini 3 capabilities.",
        session_id=session.id
    )
    
    return result.text

In [15]:
# 5. EXECUTE (In Jupyter, we await directly)
# Note: Ensure run_research is defined to return the events or the final text

async def run_research(query):
    session_service = InMemorySessionService()
    runner = Runner(
        agent=my_team, 
        app_name="ResearchApp", 
        session_service=session_service
    )
    
    session = await session_service.create_session(
        app_name="ResearchApp", 
        user_id="local_user"
    )

    print(f"🚀 Processing: {query}...")
    
    # We iterate through the stream to find the final response
    async for event in runner.run_async(input=query, session_id=session.id):
        # We look for the 'content' parts which contain the actual text
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    return part.text # Returns the final consolidated text

# Run and print
result = await run_research("What are the best performing stocks in the Indian market today, February 13, 2026?")
print("\n--- INDIAN MARKET BRIEFING ---\n")
print(result)

🚀 Processing: What are the best performing stocks in the Indian market today, February 13, 2026?...


TypeError: Runner.run_async() got an unexpected keyword argument 'input'

In [14]:
result = await run_research("What are the best performing stocks in the Indian market today, February 13, 2026?")
print("\n--- INDIAN MARKET BRIEFING ---\n")
print(result)

TypeError: run_research() takes 0 positional arguments but 1 was given

In [16]:
from google.genai import types  # Ensure you have this import

async def run_research(query):
    session_service = InMemorySessionService()
    runner = Runner(
        agent=my_team, 
        app_name="ResearchApp", 
        session_service=session_service
    )
    
    session = await session_service.create_session(
        app_name="ResearchApp", 
        user_id="local_user"
    )

    # 1. Convert string to the required Content format
    content = types.Content(
        role="user",
        parts=[types.Part(text=query)]
    )

    print(f"🚀 Processing: {query}...")
    
    # 2. FIXED: Use 'new_message' instead of 'input'
    async for event in runner.run_async(
        new_message=content, 
        session_id=session.id
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    return part.text

In [17]:
result = await run_research("What are the best performing stocks in the Indian market today, February 13, 2026?")
print("\n--- INDIAN MARKET BRIEFING ---\n")
print(result)

🚀 Processing: What are the best performing stocks in the Indian market today, February 13, 2026?...


TypeError: Runner.run_async() missing 1 required keyword-only argument: 'user_id'

In [22]:
async def run_research(query):
    session_service = InMemorySessionService()
    runner = Runner(
        agent=my_team, 
        app_name="ResearchApp", 
        session_service=session_service
    )
    
    # Define your user_id consistently
    CURRENT_USER = "local_user"
    
    session = await session_service.create_session(
        app_name="ResearchApp", 
        user_id=CURRENT_USER
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=query)]
    )

    print(f"🚀 Processing: {query}...")
    
    # FIXED: Added user_id to the run_async call
    async for event in runner.run_async(
        new_message=content, 
        session_id=session.id,
        user_id=CURRENT_USER  # <--- Added this required argument
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    return part.text

In [23]:
result = await run_research("What are the best performing stocks in the Indian market today, February 13, 2026?")
print("\n--- INDIAN MARKET BRIEFING ---\n")
print(result)

🚀 Processing: What are the best performing stocks in the Indian market today, February 13, 2026?...


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [28]:
import os
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
# If you are using Vertex AI, ensure this is True; otherwise, leave it out for AI Studio
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
# Change this for BOTH the researcher and editor
researcher = LlmAgent(
    name="Researcher",
    description="Finds technical facts about AI models.",
    instruction="List 3 key technical features of Gemini 3.",
    model="gemini-3-flash-preview"  # <--- Added '-preview'
)

editor = LlmAgent(
    name="Editor",
    description="Summarizes technical facts.",
    instruction="Turn the research into a one-sentence summary.",
    model="gemini-3-flash-preview"  # <--- Added '-preview'
)


from google.genai import types

async def run_research(query):
    session_service = InMemorySessionService()
    
    # 1. FIXED: Explicitly set the api_version to 'v1'
    runner = Runner(
        agent=my_team, 
        app_name="ResearchApp", 
        session_service=session_service,
        http_options=types.HttpOptions(api_version="v1") # <--- Add this
    )
    
    CURRENT_USER = "local_user"
    session = await session_service.create_session(
        app_name="ResearchApp", 
        user_id=CURRENT_USER
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=query)]
    )

    print(f"🚀 Processing: {query}...")
    
    async for event in runner.run_async(
        new_message=content, 
        session_id=session.id,
        user_id=CURRENT_USER
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    return part.text

In [29]:
result = await run_research("What are the best performing stocks in the Indian market today, February 13, 2026?")
print("\n--- INDIAN MARKET BRIEFING ---\n")
print(result)

TypeError: Runner.__init__() got an unexpected keyword argument 'http_options'

In [1]:
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# 1. Update Agents - Note: 'gemini-3-flash-preview' is the current Feb 2026 string
researcher = LlmAgent(
    name="Researcher",
    model="gemini-3-flash-preview",
    instruction="List the top 3 performing stocks in the Indian market for Feb 13, 2026."
)

editor = LlmAgent(
    name="Editor",
    model="gemini-3-flash-preview",
    instruction="Summarize the research into a concise market briefing."
)

my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[researcher, editor]
)

# 2. Corrected Run Function
async def run_research(query):
    session_service = InMemorySessionService()
    
    # Initialize Runner without http_options (use defaults or env vars)
    runner = Runner(
        agent=my_team, 
        app_name="ResearchApp", 
        session_service=session_service
    )
    
    CURRENT_USER = "local_user"
    session = await session_service.create_session(
        app_name="ResearchApp", 
        user_id=CURRENT_USER
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=query)]
    )

    print(f"🚀 Processing: {query}...")
    
    # The run_async call
    async for event in runner.run_async(
        new_message=content, 
        session_id=session.id,
        user_id=CURRENT_USER
    ):
        # Using the standard event check for 2026
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    # In a real app, you'd collect these or yield them
                    # For a simple print, we'll just return the final one
                    final_text = part.text
    return final_text

# 3. EXECUTE
result = await run_research("What are the best performing stocks in India today?")
print("\n--- INDIAN MARKET BRIEFING ---\n")
print(result)

🚀 Processing: What are the best performing stocks in India today?...


ValueError: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.

In [4]:
import os
from google.genai import types

async def run_research(query):
    # Initialize the session service
    session_service = InMemorySessionService()
    
    # Pass the API key explicitly to the Runner's internal client
    runner = Runner(
        agent=my_team, 
        app_name="ResearchApp", 
        session_service=session_service,
        # This tells the Runner's internal genai.Client exactly how to authenticate
        client_options={
            "api_key": os.environ.get("GOOGLE_API_KEY"),
            "vertexai": False
        }
    )
    
    CURRENT_USER = "local_user"
    session = await session_service.create_session(
        app_name="ResearchApp", 
        user_id=CURRENT_USER
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=query)]
    )

    print(f"🚀 Processing: {query}...")
    
    async for event in runner.run_async(
        new_message=content, 
        session_id=session.id,
        user_id=CURRENT_USER
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    return part.text


In [5]:
result = await run_research("What are the best performing stocks in India today?")
print("\n--- INDIAN MARKET BRIEFING ---\n")
print(result)

TypeError: Runner.__init__() got an unexpected keyword argument 'client_options'

In [ ]:
import os

# 1. FORCE THE MODE: Set to False for AI Studio (API Key mode)
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

# 2. THE KEY: Paste your key here (keep the quotes)
os.environ["GOOGLE_API_KEY"] = "xx"

# 3. THE LOCATION: Crucial for Gemini 3
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

print("✅ Environment variables locked in.")

✅ Environment variables locked in.


In [4]:
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Define your agents normally - ADK will now pick up the API key from the environment
researcher = LlmAgent(
    name="Researcher",
    model="gemini-3-flash-preview",
    instruction="List the top 3 performing stocks in India for today, Feb 13, 2026."
)

editor = LlmAgent(
    name="Editor",
    model="gemini-3-flash-preview",
    instruction="Summarize the stock info into a brief headline."
)

my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[researcher, editor]
)

async def run_research(query):
    session_service = InMemorySessionService()
    
    # SIMPLE RUNNER: No client_options, no http_options
    # It will automatically find GOOGLE_API_KEY and use it
    runner = Runner(
        agent=my_team, 
        app_name="ResearchApp", 
        session_service=session_service
    )
    
    session = await session_service.create_session(
        app_name="ResearchApp", 
        user_id="local_user"
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=query)]
    )

    print(f"🚀 Processing: {query}...")
    
    async for event in runner.run_async(
        new_message=content, 
        session_id=session.id,
        user_id="local_user"
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    return part.text

# RUN IT
result = await run_research("What are the best performing stocks in India today, 13 Feb 2026? Ensure correct market information is taken into account. Use authentic web sources to retrieve market data")
print("\n--- INDIAN MARKET BRIEFING ---\n")
print(result)

🚀 Processing: What are the best performing stocks in India today, 13 Feb 2026? Ensure correct market information is taken into account. Use authentic web sources to retrieve market data...

--- INDIAN MARKET BRIEFING ---

As an AI, I cannot provide real-time or historical market data for a future date. Today's date is currently in **May 2024**, and therefore **February 13, 2026**, has not yet occurred.

Stock market performance is highly volatile and depends on real-time economic factors, corporate earnings, and global events that cannot be predicted with certainty.

To find the top-performing stocks on any given day once that date arrives, I recommend checking authentic financial news sources and market trackers such as:

1.  **NSE (National Stock Exchange of India) Official Website:** Look for the "Top Gainers" section.
2.  **Moneycontrol:** A leading financial platform for live Indian market updates.
3.  **Economic Times (ET Markets):** For detailed analysis of daily market leaders.

In [3]:
result = await run_research("What are the best performing stocks in India today, 13 Feb 2026?")
print("\n--- INDIAN MARKET BRIEFING ---\n")
print(result)

🚀 Processing: What are the best performing stocks in India today, 13 Feb 2026?...

--- INDIAN MARKET BRIEFING ---

As a researcher, I must inform you that I cannot provide stock market data for **February 13, 2026**, as that date is in the future. Stock market performance is determined by real-time trading activity, economic reports, and corporate news that have not yet occurred.

If you are looking for historical performance data from a past date or current trends for the present day, please let me know, and I will be happy to assist you with that information.


In [5]:
from google.adk.tools import AgentTool

# 1. Create a specialized Searcher
search_specialist = LlmAgent(
    name="SearchSpecialist",
    model="gemini-3-flash-preview",
    instruction="Search the web for real-time Indian stock market gainers and losers.",
    tools=[google_search]
)

# 2. Wrap the specialist as a tool for your Main Team
my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[
        AgentTool(agent=search_specialist), # The team now 'calls' the searcher
        editor 
    ]
)


NameError: name 'google_search' is not defined

In [6]:
# 1. Correct Imports
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools import AgentTool, GoogleSearchTool  # Note the class name

# 2. Create the specialized Searcher
# IMPORTANT: Use GoogleSearchTool() with parentheses to create an instance
search_specialist = LlmAgent(
    name="SearchSpecialist",
    model="gemini-3-flash-preview",
    instruction="Search the web for real-time Indian stock market gainers and losers.",
    tools=[GoogleSearchTool()]  # <--- FIXED: Class instantiation
)

# 3. Wrap the specialist as a tool for your Main Team
# This allows your Editor to "call" the Searcher as if it were a function
my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[
        AgentTool(agent=search_specialist), 
        editor 
    ]
)

ImportError: cannot import name 'GoogleSearchTool' from 'google.adk.tools' (c:\Users\Paras Gulati\VS Code Python\ADK_research\.venv\Lib\site-packages\google\adk\tools\__init__.py)

In [7]:
# 1. Correct Imports for 2026
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools import google_search, AgentTool  # Use lowercase google_search

# 2. Create the specialized Searcher
search_specialist = LlmAgent(
    name="SearchSpecialist",
    model="gemini-3-flash-preview",
    instruction="Search the web for real-time Indian stock market gainers and losers for today, Feb 13, 2026.",
    tools=[google_search]  # <--- FIXED: Use the lowercase imported variable
)

# 3. Wrap the specialist as a tool for your Main Team
my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[
        AgentTool(agent=search_specialist), 
        editor 
    ]
)

ValidationError: 1 validation error for SequentialAgent
sub_agents.0
  Input should be a valid dictionary or instance of BaseAgent [type=model_type, input_value=<google.adk.tools.agent_t...t at 0x00000205BE9E18E0>, input_type=AgentTool]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type

In [8]:
import os
from google.adk.agents import LlmAgent, SequentialAgent
# UPDATED: Use the specific sub-module paths
from google.adk.tools.google_search_tool import google_search
from google.adk.tools.agent_tool import AgentTool

In [12]:
# # Create a specialist with the Search tool
# search_specialist = LlmAgent(
#     name="SearchSpecialist",
#     model="gemini-3-flash-preview",
#     instruction="""Find today's (Feb 13, 2026) top gainers/losers in the Indian stock market. 
#     Look for data from NSE (National Stock Exchange) or Moneycontrol.
#     Return the stock names and their percentage changes.""",
#     tools=[google_search] # Use the lowercase object directly
# )

# # The editor will summarize the specialist's findings
# editor = LlmAgent(
#     name="Editor",
#     model="gemini-3-flash-preview",
#     instruction="Summarize the stock market data into a 3-bullet point executive briefing."
# )

search_specialist = LlmAgent(
    name="SearchSpecialist",
    model="gemini-3-flash-preview",
    instruction="Find Indian stock gainers for Feb 13, 2026.",
    tools=[google_search],
    output_key="market_data" # This saves the search result to state
)

editor = LlmAgent(
    name="Editor",
    model="gemini-3-flash-preview",
    instruction="Summarize the info found in {market_data} into one sentence."
)

In [13]:
# # Assemble the team
# my_team = SequentialAgent(
#     name="TechInsightsTeam",
#     sub_agents=[
#         # Wrap the agent as a tool so it's treated as a discrete task
#         AgentTool(agent=search_specialist), 
#         editor 
#     ]
# )
# Assemble the team correctly
my_team = SequentialAgent(
    name="TechInsightsTeam",
    # FIXED: List the agents directly. No AgentTool wrapper here!
    sub_agents=[
        search_specialist, 
        editor 
    ]
)

In [ ]:
from tenacity import retry, stop_after_attempt, wait_random_exponential

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(5))
async def run_research_with_retry(query):
    # Your existing run_research logic here
    
    return await run_research(query)

In [15]:

async def run_research(query):
    session_service = InMemorySessionService()
    
    # SIMPLE RUNNER: No client_options, no http_options
    # It will automatically find GOOGLE_API_KEY and use it
    runner = Runner(
        agent=my_team, 
        app_name="ResearchApp", 
        session_service=session_service
    )
    
    session = await session_service.create_session(
        app_name="ResearchApp", 
        user_id="local_user"
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=query)]
    )

    print(f"🚀 Processing: {query}...")
    
    async for event in runner.run_async(
        new_message=content, 
        session_id=session.id,
        user_id="local_user"
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    return part.text


In [17]:
result = await run_research("What are the best performing stocks in India today, 13 Feb 2026?")
print("\n--- INDIAN MARKET BRIEFING ---\n")
print(result)

🚀 Processing: What are the best performing stocks in India today, 13 Feb 2026?...


_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [ ]:
import os
import asyncio
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.google_search_tool import google_search
from google.genai import types

# 1. SETUP ENVIRONMENT
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
os.environ["GOOGLE_API_KEY"] = "xx" # Replace with your key
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

# 2. DEFINE AGENTS
# Researcher uses the live search tool
search_specialist = LlmAgent(
    name="SearchSpecialist",
    model="gemini-2.5-flash-lite",
    instruction="""Find today's (Feb 13, 2026) top gainers/losers in the Indian stock market. 
    Use the Google Search tool. Focus on NSE and Moneycontrol data.""",
    tools=[google_search],
    output_key="market_data"
)

# Editor summarizes findings
editor = LlmAgent(
    name="Editor",
    model="gemini-2.5-flash-lite",
    instruction="Summarize the stock data found in {market_data} into a clear executive brief."
)

# The Sequential Team
my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[search_specialist, editor]
)

# 3. DEFINE RETRY LOGIC
# We specifically want to retry if we see a '429' in the error message
def is_rate_limit_error(exception):
    return "429" in str(exception) or "RESOURCE_EXHAUSTED" in str(exception)

@retry(
    retry=retry_if_exception(is_rate_limit_error),
    wait=wait_random_exponential(min=1, max=60), # Wait 1s, 2s, 4s... up to 60s
    stop=stop_after_attempt(5) # Give up after 5 tries
)
async def run_research_with_retry(query):
    session_service = InMemorySessionService()
    runner = Runner(
        agent=my_team, 
        app_name="StockApp", 
        session_service=session_service
    )
    
    session = await session_service.create_session(app_name="StockApp", user_id="user_1")
    content = types.Content(role="user", parts=[types.Part(text=query)])

    print(f"🚀 Attempting Research: {query}...")
    
    final_response = ""
    async for event in runner.run_async(
        new_message=content, 
        session_id=session.id,
        user_id="user_1"
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    final_response += part.text
    return final_response

# 4. EXECUTION
async def main():
    try:
        result = await run_research_with_retry("What are the best performing stocks in India today?")
        print("\n--- INDIAN MARKET BRIEFING ---\n")
        print(result)
    except Exception as e:
        print(f"\n❌ Permanent Failure: {e}")

await main()

🚀 Attempting Research: What are the best performing stocks in India today?...

--- INDIAN MARKET BRIEFING ---

On Friday, February 13, 2026, the Indian stock market experienced a broad sell-off, with both the BSE Sensex and NSE Nifty closing down over 1%.

**Top Gainers on NSE Nifty 50:**

*   Bajaj Finance: +2.57%
*   Eicher Motors: +1.54%
*   SBI Life Insurance Company: +0.60% (some sources cite slightly different percentages like +0.84% and +0.23%)
*   State Bank of India (SBI): +0.52% (some sources cite +0.29%)
*   Cipla: +0.11% (one source cites +0.12%)
*   Apollo Hospital: +0.05% (one source cites +0.59%)

**Top Losers on NSE Nifty 50:**

*   Hindalco Industries: -5.75% (some sources cite -5.74% and -6.08%)
*   Hindustan Unilever (HUL): -4.34% (one source cites -4.47%)
*   Eternal Ltd: -4.30% (one source cites -4.16%)
*   Adani Enterprises: -3.83%
*   ONGC: -3.24%

Other stocks that experienced significant declines include Muthoot Finance (down 11.80%), Alkem Laboratories (down 8